# 03 — Fine-tuning a pretrained AI-text detector (desklib, DeBERTa-v3-large) for human / ai / fusion

[`desklib/ai-text-detector-v1.01`](https://huggingface.co/desklib/ai-text-detector-v1.01) is `microsoft/deberta-v3-large`
(~0.4B parameters) fine-tuned as a **binary** English human-vs-AI detector (model card lists the RAID dataset; MIT licence).
This notebook:

1. **Zero-shot check** — the published binary model on part of the fixed validation split: does its P(ai) already
   separate the classes, per language?
2. **3-class head** — the 1-logit head becomes a 3-logit head initialised from it (human = −½·logit, ai = +½·logit,
   fusion = 0), so training starts from the detector's existing human/ai decision.
3. **Fine-tuning** on a stratified subsample of the fixed training split; evaluation on the fixed validation split
   (macro-F1, per class, per domain/language), test submission, scores saved for ensembling.

**Long texts:** head + tail truncation — first `HEAD_TOKENS` and last `MAX_LEN − 2 − HEAD_TOKENS` tokens.

**Before relying on it, check two things:**
* *Rules.* Pretrained weights are allowed, but this model was itself trained on an external labelled AI-detection
  dataset. Confirm with the course staff that this counts as "pretrained weights", and state it in the report.
* *Language.* The model is English-only; ~17% of texts are German. Per-language results below show the cost.

**Compute (estimates, not measured):** needs a 16 GB GPU (Kaggle/Colab T4, or the RX 6800 XT with ROCm). Full
fine-tuning of 0.4B parameters with AdamW needs ~7 GB for weights + gradients + optimizer state alone, so an 8 GB
card (RTX 4060 laptop) needs `FREEZE_LAYERS ≈ 12`. On a T4, 20k texts × 2 epochs at 512 tokens is roughly 3–4 h.
Run with `TRAIN_N = 500, EPOCHS = 1` first to check memory and speed.

In [1]:
import os
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    # Kaggle: attach the competition data + a dataset containing this repo's src/ and splits/ folders
    inp = Path("/kaggle/input")
    os.environ["HW3_DATA_DIR"] = str(next(inp.rglob("train.csv")).parent)
    os.environ["HW3_SPLITS_DIR"] = str(next(inp.rglob("val_ids.csv")).parent)
    os.environ["HW3_OUTPUT_DIR"] = "/kaggle/working/outputs"
    os.environ["HW3_SUBMISSION_DIR"] = "/kaggle/working/submissions"
    SRC = next(inp.rglob("config.py")).parent
else:
    SRC = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src") if (q / "config.py").exists())
sys.path.insert(0, str(SRC))

In [2]:
import gc
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoModel, AutoTokenizer, PreTrainedModel

from config import LABELS, OUTPUT_DIR, SEED
from utils import load_splits, load_test, log_result, macro_f1, save_submission, save_val_scores

MODEL_ID = "desklib/ai-text-detector-v1.01"
MAX_LEN = 512             # tokens incl. special tokens (desklib's example uses 768; 512 roughly halves compute)
HEAD_TOKENS = 128         # long texts keep the first 128 and the last MAX_LEN-2-128 tokens
TRAIN_N = 20_000          # stratified subsample of the fixed training split (compute budget)
EPOCHS = 2
LR_BACKBONE, LR_HEAD = 1e-5, 1e-4
BATCH, ACCUM = 4, 4       # effective batch 16; if out of GPU memory: BATCH 2, ACCUM 8
WARMUP = 0.1              # share of optimizer steps with linear warm-up
WEIGHT_DECAY = 0.01
CLASS_WEIGHTS = True      # inverse-frequency loss weights (macro-F1 counts every class equally)
FREEZE_LAYERS = 0         # freeze embeddings + first N of 24 layers; ~12 on an 8 GB GPU (estimate)
GRAD_CHECKPOINTING = True # recompute activations in backward: less memory, ~30% slower
ZERO_SHOT_N = 3_000       # validation texts for the zero-shot check (0 = skip)
EVAL_BATCH = 16
LOG_EVERY = 50            # optimizer steps between progress lines

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16 = DEVICE == "cuda"   # T4 has no bf16; if the loss turns NaN, set FP16 = False
CKPT = OUTPUT_DIR / "03_desklib_best.pt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"device={DEVICE} {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''}  fp16={FP16}")

device=cuda AMD Radeon RX 6800 XT  fp16=True


## Model class and data helpers

In [3]:
class DesklibAIDetectionModel(PreTrainedModel):
    """Same layout as on the model card (self.model + self.classifier, mean pooling), so the published weights load."""
    config_class = AutoConfig

    def __init__(self, config):
        super().__init__(config)
        self.model = AutoModel.from_config(config)
        self.classifier = nn.Linear(config.hidden_size, 1)
        self.post_init()

    def forward(self, input_ids, attention_mask):
        h = self.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)      # mean over real (non-padding) tokens
        return self.classifier(pooled)


tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.model_max_length = 10**9          # we truncate ourselves; silences the "sequence too long" warning


def encode(texts: pd.Series) -> list:
    """Token ids with head+tail truncation to MAX_LEN (texts cut at 20k chars first to bound tokenizer time)."""
    body = MAX_LEN - 2
    tail = body - HEAD_TOKENS
    ids = tok(texts.astype(str).str.slice(0, 20_000).tolist(), add_special_tokens=False)["input_ids"]
    cls, sep = tok.cls_token_id, tok.sep_token_id        # DeBERTa input format: [CLS] tokens [SEP]
    return [[cls] + (x if len(x) <= body else x[:HEAD_TOKENS] + x[-tail:]) + [sep] for x in ids]


def batches(seqs, labels=None, bs=8, rng=None):
    """Yield (row indices, input_ids, attention_mask, labels); shuffled if rng is given, else sorted by length."""
    order = rng.permutation(len(seqs)) if rng is not None else np.argsort([len(s) for s in seqs])
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        L = max(len(seqs[j]) for j in idx)
        ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, j in enumerate(idx):
            ids[r, :len(seqs[j])] = torch.tensor(seqs[j])
            att[r, :len(seqs[j])] = 1
        yield idx, ids, att, (torch.as_tensor(labels[idx]) if labels is not None else None)


@torch.no_grad()
def predict_logits(model, seqs) -> np.ndarray:
    model.eval()
    out = np.zeros((len(seqs), model.classifier.out_features), dtype=np.float32)
    for idx, ids, att, _ in batches(seqs, bs=EVAL_BATCH):
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            out[idx] = model(ids.to(DEVICE), att.to(DEVICE)).float().cpu().numpy()
    return out

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [4]:
train_df, val_df = load_splits()
test_df = load_test()
strata = train_df.domain + "|" + train_df.label
train_sub = (train_test_split(train_df, train_size=TRAIN_N, stratify=strata, random_state=SEED)[0]
             if TRAIN_N < len(train_df) else train_df).reset_index(drop=True)
lab2id = {l: i for i, l in enumerate(LABELS)}
y_tr = train_sub.label.map(lab2id).to_numpy()
y_va = val_df.label.to_numpy()

t0 = time.time()
tr_seqs, va_seqs = encode(train_sub.text), encode(val_df.text)
lens = np.array([len(s) for s in tr_seqs])
print(f"train={len(train_sub):,}  val={len(val_df):,}  tokenized in {time.time() - t0:.0f}s; "
      f"{(lens >= MAX_LEN).mean():.0%} of training texts hit MAX_LEN={MAX_LEN} (head+tail truncated)")

train=20,000  val=10,000  tokenized in 8s; 60% of training texts hit MAX_LEN=512 (head+tail truncated)


## 1. Zero-shot: the published binary detector as it is

In [5]:
model = DesklibAIDetectionModel.from_pretrained(MODEL_ID).to(DEVICE)

if ZERO_SHOT_N:
    zs = val_df.sample(min(ZERO_SHOT_N, len(val_df)), random_state=SEED).reset_index(drop=True)
    p_ai = 1 / (1 + np.exp(-predict_logits(model, encode(zs.text))[:, 0]))
    zs["p_ai"] = p_ai

    def auc(pos, neg, mask=slice(None)):
        d = zs[mask]; d = d[d.label.isin([pos, neg])]
        return roc_auc_score(d.label == pos, d.p_ai) if d.label.nunique() == 2 else np.nan

    rows = [{"subset": name, "AUC ai vs human": auc("ai", "human", m), "AUC fusion vs human": auc("fusion", "human", m),
             "AUC ai vs fusion": auc("ai", "fusion", m)}
            for name, m in [("all", slice(None))] + [(l, zs.lang == l) for l in sorted(zs.lang.unique())]]
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    print("\nMedian P(ai) by language and class:")
    print(zs.pivot_table(index="lang", columns="label", values="p_ai", aggfunc="median")[LABELS].round(3).to_string())

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[W1124 21:17:35.456044246 HIPCachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 264241152 bytes (free: 138412032, total: 17163091968).


subset  AUC ai vs human  AUC fusion vs human  AUC ai vs fusion
   all            0.977                0.739             0.875
 de-DE            0.990                0.652             0.950
 en-EN            0.994                0.810             0.932

Median P(ai) by language and class:
label  human     ai  fusion
lang                       
de-DE  0.005  0.686   0.010
en-EN  0.052  1.000   0.863


## 2. Binary head → 3-class head

softmax over (−s/2, +s/2, 0) gives P(ai)/(P(ai)+P(human)) = sigmoid(s), i.e. the original detector's decision between
human and ai, with fusion starting neutral.

In [6]:
old = model.classifier
head = nn.Linear(model.config.hidden_size, len(LABELS))
with torch.no_grad():
    head.weight.zero_(); head.bias.zero_()
    head.weight[lab2id["ai"]] = old.weight[0] / 2;     head.bias[lab2id["ai"]] = old.bias[0] / 2
    head.weight[lab2id["human"]] = -old.weight[0] / 2; head.bias[lab2id["human"]] = -old.bias[0] / 2
model.classifier = head.to(DEVICE)
model.config.num_labels = len(LABELS)

if FREEZE_LAYERS:
    for p in model.model.embeddings.parameters():
        p.requires_grad = False
    for layer in model.model.encoder.layer[:FREEZE_LAYERS]:
        for p in layer.parameters():
            p.requires_grad = False
if GRAD_CHECKPOINTING:
    model.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_train / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

trainable parameters: 434M of 434M


/home/yagodanr-ubuntu/UCU/AI/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:3204: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


## 3. Fine-tuning

In [7]:
no_decay = lambda n: n.endswith("bias") or "LayerNorm" in n
backbone = [(n, p) for n, p in model.named_parameters() if p.requires_grad and n.startswith("model.")]
opt = torch.optim.AdamW([
    {"params": [p for n, p in backbone if not no_decay(n)], "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in backbone if no_decay(n)], "lr": LR_BACKBONE, "weight_decay": 0.0},
    {"params": list(model.classifier.parameters()), "lr": LR_HEAD, "weight_decay": 0.0},
])
total_steps = EPOCHS * math.ceil(len(tr_seqs) / (BATCH * ACCUM))
warm = max(1, int(WARMUP * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / max(1, total_steps - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=FP16)

counts = np.bincount(y_tr, minlength=len(LABELS))
weights = torch.tensor(counts.sum() / (len(LABELS) * counts), dtype=torch.float32) if CLASS_WEIGHTS else None
loss_fn = nn.CrossEntropyLoss(weight=weights.to(DEVICE) if weights is not None else None)
print(f"optimizer steps: {total_steps}  warm-up: {warm}  class weights: "
      f"{dict(zip(LABELS, weights.numpy().round(2))) if weights is not None else None}")

optimizer steps: 2500  warm-up: 250  class weights: {'human': np.float32(1.38), 'ai': np.float32(0.72), 'fusion': np.float32(1.12)}


In [ ]:
rng = np.random.default_rng(SEED)
best_f1, step, history = -1.0, 0, []
t_start = time.time()
for epoch in range(EPOCHS):
    model.train()
    running, n_seen = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for b, (idx, ids, att, yb) in enumerate(batches(tr_seqs, y_tr, bs=BATCH, rng=rng)):
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            logits = model(ids.to(DEVICE), att.to(DEVICE))
        loss = loss_fn(logits.float(), yb.to(DEVICE)) / ACCUM
        if not torch.isfinite(loss):
            raise RuntimeError("loss is NaN/inf - set FP16 = False or lower LR_BACKBONE")
        scaler.scale(loss).backward()
        running += loss.item() * ACCUM; n_seen += 1
        if (b + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % LOG_EVERY == 0:
                el = time.time() - t_start
                print(f"epoch {epoch + 1} step {step}/{total_steps}  loss {running / n_seen:.4f}  "
                      f"lr {sched.get_last_lr()[0]:.2e}  elapsed {el / 60:.0f} min  ETA {el / step * (total_steps - step) / 60:.0f} min")
                running, n_seen = 0.0, 0

    va_logits = predict_logits(model, va_seqs)
    f1 = macro_f1(y_va, np.array(LABELS)[va_logits.argmax(1)])
    history.append({"epoch": epoch + 1, "val_macro_f1": f1, "minutes": (time.time() - t_start) / 60})
    print(f"== epoch {epoch + 1}: validation macro-F1 = {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), CKPT)
        np.save(OUTPUT_DIR / "03_desklib_val_logits.npy", va_logits)
print(pd.DataFrame(history).round(4).to_string(index=False))

epoch 1 step 50/2500  loss 0.9089  lr 2.04e-06  elapsed 2 min  ETA 95 min


## 4. Evaluation of the best epoch (fixed validation split)

In [ ]:
va_logits = np.load(OUTPUT_DIR / "03_desklib_val_logits.npy")
va_prob = torch.softmax(torch.tensor(va_logits), 1).numpy()
pred_va = np.array(LABELS)[va_prob.argmax(1)]
print(f"macro-F1 = {macro_f1(y_va, pred_va):.4f}\n")
print(classification_report(y_va, pred_va, labels=LABELS, digits=3))
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(confusion_matrix(y_va, pred_va, labels=LABELS), index=LABELS, columns=LABELS).to_string())

ev = val_df[["domain", "lang"]].assign(y=y_va, pred=pred_va)
rows = [{"group": col, "value": v, "n": len(g), "macro_f1": macro_f1(g.y, g.pred)}
        for col in ("lang", "domain") for v, g in ev.groupby(col)]
print("\n" + pd.DataFrame(rows).round(3).to_string(index=False))

log_result("03_encoder", f"desklib deberta-v3-large ft | n={len(train_sub)} ep={EPOCHS} len={MAX_LEN} head+tail",
           y_va, pred_va, notes="pretrained detector, 3-class head")
save_val_scores("03_desklib_ft", val_df.id, va_prob, LABELS)

## 5. Test predictions and comparison with the baseline

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
te_prob = torch.softmax(torch.tensor(predict_logits(model, encode(test_df.text))), 1).numpy()
(OUTPUT_DIR / "test_scores").mkdir(exist_ok=True)
pd.DataFrame(te_prob, columns=LABELS).assign(id=test_df.id.values).to_csv(OUTPUT_DIR / "test_scores" / "03_desklib_ft.csv", index=False)
save_submission(test_df.id, np.array(LABELS)[te_prob.argmax(1)], "03_desklib_ft")

res = pd.read_csv(OUTPUT_DIR / "results.csv")
best_per_exp = res.loc[res.groupby("experiment").macro_f1.idxmax()]
print(best_per_exp[["experiment", "model", "macro_f1", "f1_human", "f1_ai", "f1_fusion"]].to_string(index=False))